In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

df = pd.read_excel("/kaggle/input/datasets/muratkokludataset/dry-bean-dataset/Dry_Bean_Dataset/Dry_Bean_Dataset.xlsx")

X = df.drop(columns=["Class"])
encoder = LabelEncoder()
y = encoder.fit_transform(df["Class"])

X = torch.tensor(X.values, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [2]:

# MLP: 16 признаков → 64 нейрона → 7 классов
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(16, 64),
            nn.ReLU(),
            nn.Linear(64, 7)
        )

    def forward(self, x):
        return self.layers(x)


# model
model = MLP()

# loss
criterion = nn.CrossEntropyLoss()

# optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)

# DataLoader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)


def train_one_epoch(model, train_loader, criterion, optimizer):
    model.train()

    total_loss = 0.0

    for X_batch, y_batch in train_loader:
        # 1. Сбрасываем градиенты предыдущего батча
        optimizer.zero_grad()

        # 2. Forward pass → получаем logits
        logits = model(X_batch)

        # 3. Считаем loss
        loss = criterion(logits, y_batch)

        # 4. Backward pass → вычисляем градиенты
        loss.backward()

        # 5. Обновляем веса
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)


# Одна эпоха
loss = train_one_epoch(
    model,
    train_loader,
    criterion,
    optimizer
)

print(f"Loss: {loss:.4f}")

Loss: 368.3182
